## Demo de la aplicación: del modelo al usuario final

### By:
Bryan Escobar Restrepo

### Date:
2026-08-21

### Description:

Octava y última etapa (`7-deploy`). Documenta la **demo funcional** del modelo: un
formulario web donde un aspirante introduce su perfil y recibe su probabilidad estimada de
admisión, construido con [Streamlit](https://streamlit.io/), siguiendo el enfoque de
<https://joserzapata.github.io/post/ciencia-datos-proyecto-python/8-deploy/>.

Este notebook **no** es la aplicación: la aplicación es `app.py`. Aquí se documenta cómo
está construida, se verifica que el artefacto funciona fuera de los notebooks y se dejan
las instrucciones de ejecución y despliegue.

**El principio que guía el diseño:** la demo no muestra un número y ya. Un producto que
aconseja a personas sobre decisiones caras —postular a un posgrado cuesta dinero y un año
de vida— tiene que comunicar también **qué tan seguro está** y **por qué** dice lo que
dice. Las tres limitaciones que encontró `07-interpretation` están implementadas como
comportamiento de la interfaz, no como letra pequeña.

## 🏗️ Arquitectura de la demo

```text
app.py                          interfaz de Streamlit (formulario, resultados, gráficos)
└── src/inference/prediccion.py lógica: cargar, predecir, clasificar, explicar
    └── data/06_models/modelo_final_automl.joblib
        ├── preparacion  → pipeline de 04-feat_eng (imputación, escalado, encoding)
        └── modelo       → Extra Trees seleccionado por AutoML en 06
```

**Por qué la lógica no vive dentro de `app.py`:** una interfaz rota se ve a simple vista,
pero una regla de negocio mal escrita —el corte entre "probable" y "segura", el umbral de
la advertencia— no se ve, y es justo la que hace daño. Separándola en `src/inference/` se
puede probar con `pytest`, y de hecho hay **9 pruebas** que la cubren, incluidas tres que
ejecutan la aplicación completa y simulan a un usuario rellenando el formulario.

| Archivo | Qué hace |
|---|---|
| `app.py` | formulario, resultados, gráfico de aportes, advertencias |
| `src/inference/prediccion.py` | carga del modelo, predicción, cestas, intervalos, SHAP |
| `tests/test_prediccion.py` | 6 pruebas de la lógica de inferencia |
| `tests/test_app.py` | 3 pruebas de extremo a extremo de la interfaz |
| `requirements.txt` | dependencias **fijadas** para el despliegue |
| `packages.txt` | `libgomp1`, la librería de sistema que necesita scikit-learn |

## 📚 Import  libraries

In [1]:
# base libraries for data science
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)

## ⚙️ Configuración

In [2]:
def buscar_raiz_proyecto() -> Path:
    """Sube por el árbol de directorios hasta encontrar la raíz del repositorio."""
    marcadores = (".git", "pyproject.toml")
    actual = Path.cwd().resolve()
    for candidato in (actual, *actual.parents):
        if any((candidato / marcador).exists() for marcador in marcadores):
            return candidato
    return actual


RAIZ = buscar_raiz_proyecto()
sys.path.insert(0, str(RAIZ / "src"))

from inference.prediccion import (  # noqa: E402
    CORTES_CESTA,
    MAE_MODELO,
    UMBRAL_ADVERTENCIA,
    cargar_modelo,
    clasificar_cesta,
    contribuciones,
    intervalo_estimado,
    predecir,
    prediccion_media,
)

print(f"Cortes de cesta      : {CORTES_CESTA}")
print(f"MAE del modelo       : {MAE_MODELO}")
print(f"Umbral de advertencia: {UMBRAL_ADVERTENCIA}")

Cortes de cesta      : (0.67, 0.79)
MAE del modelo       : 0.0476
Umbral de advertencia: 0.55


## ✅ El artefacto funciona fuera de los notebooks

Lo primero que hay que comprobar antes de desplegar nada: que el `.joblib` se carga en un
proceso limpio, sin depender de variables que quedaran en memoria de otro cuaderno, y que
acepta **datos crudos** tal como los va a escribir un usuario en un formulario — con sus
enteros, sus decimales y sus casillas sin rellenar.

In [3]:
modelo = cargar_modelo()
print(f"Pipeline cargado: {[nombre for nombre, _ in modelo.steps]}")
print(f"Modelo          : {type(modelo.named_steps['modelo']).__name__}")
print(f"Prediccion media: {prediccion_media(modelo):.4f}")

Pipeline cargado: ['preparacion', 'modelo']
Modelo          : ExtraTreesRegressor
Prediccion media: 0.7221


In [4]:
PERFILES = {
    "aspirante fuerte": {
        "gre_score": 335,
        "toefl_score": 118,
        "university_rating": 5.0,
        "sop": 5.0,
        "lor": 4.5,
        "cgpa": 9.6,
        "research": 1.0,
    },
    "aspirante promedio": {
        "gre_score": 316,
        "toefl_score": 107,
        "university_rating": 3.0,
        "sop": 3.5,
        "lor": 3.5,
        "cgpa": 8.6,
        "research": 1.0,
    },
    "aspirante debil": {
        "gre_score": 295,
        "toefl_score": 95,
        "university_rating": 1.0,
        "sop": 2.0,
        "lor": 2.0,
        "cgpa": 7.2,
        "research": 0.0,
    },
    "sin TOEFL ni LOR": {
        "gre_score": 320,
        "toefl_score": None,
        "university_rating": 4.0,
        "sop": 4.0,
        "lor": None,
        "cgpa": 9.0,
        "research": 1.0,
    },
}

filas = []
for nombre, perfil in PERFILES.items():
    probabilidad = predecir(modelo, perfil)
    inferior, superior = intervalo_estimado(probabilidad)
    filas.append(
        {
            "perfil": nombre,
            "probabilidad": probabilidad,
            "rango": f"{inferior:.0%} a {superior:.0%}",
            "cesta": clasificar_cesta(probabilidad),
            "muestra_advertencia": probabilidad < UMBRAL_ADVERTENCIA,
        }
    )

pd.DataFrame(filas).set_index("perfil").round(3)

,probabilidad,rango,cesta,muestra_advertencia
perfil,,,,
aspirante fuerte,0.937,89% a 98%,segura,False
aspirante promedio,0.740,69% a 79%,probable,False
aspirante debil,0.478,43% a 53%,ambiciosa,True
sin TOEFL ni LOR,0.802,75% a 85%,segura,False


Los cuatro casos se comportan como deben:

- El orden entre perfiles es el esperado, y cada uno cae en la cesta que le corresponde.
- **El perfil incompleto funciona igual.** Un aspirante que no recuerda su TOEFL obtiene su
  estimación: el pipeline imputa con la mediana aprendida en entrenamiento. Es una decisión
  de producto deliberada — obligar a rellenar todo habría hecho abandonar el formulario a
  quien no tiene los datos a mano.
- El perfil débil **activa la advertencia**, que es la salvaguarda que pedía
  `07-interpretation`.

## 🧠 La explicación que ve el usuario

`07-interpretation` propuso mostrar la explicación SHAP al usuario final, porque en un
producto de orientación *"tu promedio suma 4 puntos, no tener investigación te resta 3"*
vale más que un número solo. La demo lo implementa: cada predicción viene con el desglose
de qué aportó cada dato.

In [5]:
perfil_ejemplo = PERFILES["aspirante promedio"]
aportes = contribuciones(modelo, perfil_ejemplo)

desglose = pd.DataFrame(
    {
        "aporte": aportes,
        "efecto": ["sube la probabilidad" if valor >= 0 else "la baja" for valor in aportes],
    }
)
print(f"Punto de partida (media de todos los aspirantes): {prediccion_media(modelo):.1%}")
print(f"Suma de los aportes                             : {aportes.sum():+.1%}")
print(f"Probabilidad final                              : {predecir(modelo, perfil_ejemplo):.1%}")
desglose.round(4)

Punto de partida (media de todos los aspirantes): 72.2%
Suma de los aportes                             : +1.8%
Probabilidad final                              : 74.0%


,aporte,efecto
research,0.0183,sube la probabilidad
toefl_score,-0.0045,la baja
gre_score,-0.0039,la baja
sop,0.0039,sube la probabilidad
university_rating,0.0037,sube la probabilidad
cgpa,0.0004,sube la probabilidad
lor,0.0004,sube la probabilidad


La descomposición es **exacta**, no aproximada: la media más la suma de los aportes da
justo la predicción. Esa propiedad de SHAP es la que permite enseñársela a un usuario sin
que las cifras dejen de cuadrar.

## 🛡️ Las tres salvaguardas implementadas

Cada una responde a un hallazgo medido en `07-interpretation`, y está en el código de la
interfaz, no en un aviso legal que nadie lee:

| Hallazgo de `07` | Qué hace la interfaz |
|---|---|
| El modelo **no predice por debajo de 0.45** y sobrestima en el tramo bajo | Muestra una advertencia explícita cuando la predicción baja de 0.55, diciendo que la probabilidad real podría ser menor |
| El MAE es 0.0476: el número tiene margen | Junto a la cifra muestra un **rango de referencia** de ±1 MAE, no solo el punto |
| El modelo es **más fiable ordenando que dando la cifra exacta** (80 % de acierto de cesta) | El resultado principal es la cesta (segura / probable / ambiciosa), con la probabilidad como dato secundario |

Y una cuarta, de honestidad general: un desplegable explica los límites del modelo con
números concretos —el sesgo de la muestra, el rango que no cubre, el ruido del propio
objetivo— y recuerda que no sustituye a un asesor académico.

## ▶️ Cómo ejecutar la demo en local

```bash
# 1. instalar dependencias (incluye streamlit)
uv sync --all-extras --dev

# 2. levantar la aplicación
uv run streamlit run app.py
```

Se abre en <http://localhost:8501>. No hace falta ejecutar ningún notebook antes: el
modelo ya está versionado en `data/06_models/modelo_final_automl.joblib`.

Para comprobar que todo está bien sin abrir el navegador:

```bash
uv run pytest tests/test_app.py tests/test_prediccion.py -v
```

## ☁️ La demo publicada

**La demo está desplegada en <https://admisiones-project-cd.streamlit.app/>**

Se publicó en [Streamlit Community Cloud](https://share.streamlit.io) (gratuito, se conecta
al repositorio de GitHub) con estos pasos:

1. Entrar en <https://share.streamlit.io> con la cuenta de GitHub.
2. **Create app** → **Deploy a public app from GitHub**.
3. Rellenar:
   - Repository: `bryanescobarr/Admisiones-project`
   - Branch: `main`
   - Main file path: `app.py`
   - En *Advanced settings*, Python version: **3.12**
4. **Deploy**. La primera compilación tarda unos minutos.

Tres detalles que hacen que el despliegue funcione a la primera:

- **`requirements.txt` con versiones fijadas.** El modelo se serializó con
  `scikit-learn 1.9.0` y `pandas 3.0.5`; con otras versiones el `.joblib` puede no
  deserializarse. Streamlit Cloud lee este archivo, no el `pyproject.toml`.
- **`packages.txt` con `libgomp1`.** Es la librería de OpenMP que necesita scikit-learn;
  sin ella el contenedor falla al importar.
- **El modelo está versionado en el repositorio.** Ocupa 1.1 MB, muy por debajo de
  cualquier límite, así que la aplicación no necesita descargarlo de ningún sitio.

## 📊 Analysis of Results and Conclusions

**La demo cierra el ciclo del proyecto:** los datos crudos del PR inicial llegan hasta un
formulario donde una persona escribe su perfil y obtiene una recomendación, pasando por el
mismo pipeline de preprocesamiento y el mismo modelo que se validaron en las etapas
anteriores. No hay reimplementación de la lógica de transformación: el artefacto `.joblib`
contiene el pipeline completo, así que **lo que ocurre en producción es exactamente lo que
ocurrió al entrenar**.

**Lo que se decidió y por qué:**

- **Permitir campos vacíos.** El pipeline imputa, así que exigirlo todo habría sido una
  restricción artificial que solo servía para perder usuarios.
- **La cesta por encima de la probabilidad.** El modelo acierta la clasificación el 80 % de
  las veces y la cifra exacta con un margen del 4.8 %; la interfaz refleja esa diferencia de
  fiabilidad en vez de esconderla.
- **Advertir en lugar de ocultar.** Habría sido más cómodo no mostrar nada por debajo de
  0.55. Mostrar el número con una advertencia clara respeta más al usuario y es más honesto
  con lo que el modelo sabe y no sabe.
- **Lógica separada y probada.** 9 pruebas cubren la inferencia y la interfaz, incluida una
  que simula a un usuario rellenando el formulario con el peor perfil posible para
  comprobar que salta la advertencia.

**Limitación asumida:** la demo predice para un perfil de aspirante, no para una
universidad concreta. El dataset no tiene identificador de universidad —solo un rating
anónimo de 1 a 5—, así que el producto real necesitaría el dato que ya se pidió en
`03-analysis`: el ranking o nombre real de cada institución.

## 💡 Proposals and Ideas

- **Añadir el intervalo real** cuando se haga el experimento 2 de `07`: hoy el rango es
  ±1 MAE, una aproximación honesta pero no un intervalo de predicción calibrado.
- **Comparar varias universidades a la vez**, que es el uso real: introducir el perfil una
  vez y ver la lista ordenada en las tres cestas. Requiere el dato de universidad.
- **Registrar las consultas** (de forma anónima) para detectar deriva: si empiezan a llegar
  perfiles muy distintos a los del entrenamiento, el modelo hay que reajustarlo.
- **Mostrar qué cambiaría el resultado**: "con 0.3 más de promedio pasarías de ambiciosa a
  probable". Es la pregunta que un aspirante realmente quiere responder, y los valores SHAP
  ya dan casi todo lo necesario.

## 📖 References

- Despliegue, Jose R. Zapata:
  <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/8-deploy/>
- Streamlit: <https://docs.streamlit.io/>
- Pruebas de aplicaciones Streamlit (`AppTest`):
  <https://docs.streamlit.io/develop/api-reference/app-testing>
- Streamlit Community Cloud: <https://docs.streamlit.io/deploy/streamlit-community-cloud>
- Código de la demo: `app.py` y `src/inference/prediccion.py`